<h1><font color="#113D68" size=6>Deep Learning en Python con PyTorch</font></h1>

<h1><font color="#113D68" size=5>Parte 6. Redes Avanzadas</font></h1>

<h1><font color="#113D68" size=4>3. Transfer Lerarning</font></h1>

<br><br>
<div style="text-align: right">
<font color="#113D68" size=3>Manuel Castillo Cara</font><br>

</div>

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i>
More information about [Manuel Castillo-Cara](https://www.manuelcastillo.eu/)

<div class="alert alert-block alert-info">

<i class="fa fa-info-circle" aria-hidden="true"></i>
Puedes ver más cursos de Inteligencia Artificial, Machine Learning y Deep Learning en mi [página web](https://www.manuelcastillo.eu/udemy/)


---

<a id="indice"></a>
<h2><font color="#004D7F" size=6>Índice</font></h2>

* [1. Definición, consejos y problemas](#section1)
    * [1.1. Ejemplo](#section12)
    * [1.2. Finnetunning](#section12)
    * [1.3. Transfer Learning](#section13)
* [2. Dataset: Flickr Style](#section2)
    * [2.1. Preparación del dataset](#section21)
    * [2.2. Procesamiento de datos](#section22)
* [3. ResNet](#section3)
    * [3.1. Cargar la red](#section32)
    * [3.2. Entrenar el modelo](#section32)
* [4. Transfer Learning](#section4)
* [5. Finetuning](#section5)

<a id="section1"></a>
# <font color="#004D7F" size=6> 1. Definición, consejos y problemas</font>

¿Qué ocurre cuando no disponemos de datos suficientes o de los recursos necesarios? ¿Es posible utilizar deep learning?
1. Si la cantidad de datos que tenemos es pequeña, entrenar una red neuronal es una tarea complicada. 
2. Si reducimos el tamaño de la red, los resultados que obtengamos pueden que no sean del todo satisfactorios, ya que se va a tratar de un modelo relativamente simple. 
3. Si la red es demasiado grande, los pesos no se ajustarán adecuadamente con los pocos datos de entrenamiento y puede que incluso sobreajustemos a dicha información.

Otra de las limitaciones de las redes neuronales son los tiempos de entrenamiento. Cuando contamos con suficientes datos como para entrenar una red, podemos permitirnos aumentar el tamaño de esta. Sin embargo, esto aumenta el tiempo de los procesos `forward` y `backward` y la complejidad para ajustar los hiperparámetros. 

Por todos estos motivos surge el **Transfer Learning**, donde las redes neuronales aprendidas en problemas en los que hay más disponibilidad de datos se utilizan para resolver otros problemas para los que no fueron diseñadas.

<a id="section11"></a>
# <font color="#004D7F" size=5> 1.1. Ejemplo</font>

Un ejemplo de esto es AlexNet, representada en la imagen, que aprende propiedades de las imágenes que van desde lo más general en las primeras capas hasta las 1.000 etiquetas en su última capa, que indican muy concretamente un objeto identificado en la imagen. 

Durante el proceso `forward` no sólo obtenemos las probabilidades de la última capa, si no que en todas las demás capas se generan una **serie de activaciones** que se pueden interpretar como un conjunto de características que describen la imagen. Estas características pueden utilizarse directamente para entrenar otros modelos como por ejemplo un regresor lineal, un arból de decisión, etc.


<img src="Img/AlexNet-CNN.png" width="550">

<a id="section12"></a>
# <font color="#004D7F" size=5> 1.2. Finnetunning</font>

Este concepto consiste en el entrenamiento parcial de redes ya entrenadas, es decir, es posible editar la estructura de parte de una red neural que ya ha sido entrenada y aprender nuevos pesos sólo para las capas modificadas. 

Esto reduce mucho el tiempo de entrenamiento y la cantidad de imágenes necesarias. Volviendo al caso de AlexNet, es posible eliminar la última capa con la salida de 1.000 etiquetas y añadir una nueva capa con la salida de nuestro problema.

<img src="Img/finetunning.png" width="550">

<a id="section13"></a>
# <font color="#004D7F" size=5> 1.3. Transfer Learning</font>

En los últimos años, el **Transfer Learning** ha permitido poder utilizar el conocimiento generado en redes neuronales entrenadas con grandes bases de datos. Además, los resultados obtenidos de estas técnicas han superado en muchos problemas el estado del arte y es un area de investigación fuertemente activa. Gracias a esto, se han desarrollado una serie de pautas que, dependiendo de la cantidad de datos del problema y de su similitud con los datos de la red entrenada, nos indicarán que estrategia de Transfer Learning es más adecuada:


|  | **Problemas similares** | **Problemas diferentes** |
| --- | --- | --- |
| **Pocos datos** | Extraer características de las últimas capas y utilizarlas en otros modelos | El peor escenario, se puede probar a obtener características de capas previas a las últimas y utilizarlas en otros modelos |
| **Nº medio de datos** | Finetunning de pocas capas | Finetunning de muchas capas |

En esta práctica vamos a ver como obtener las activaciones que provoca el paso `forward` y como utilizar dicha información para entrenar otros modelos con SciKit. También vamos a ver como realizar finetunning, para utilizar algunas de las redes neuronales más famosas en nuestros problemas.

---
<div style="text-align: right"> <font size=5> <a href="#indice"><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#004D7F"></i></a></font></div>

---

<a id="section2"></a>
# <font color="#004D7F" size=6>2. Dataset: Flickr Style</font>

En posts anteriores hemos introducido la arquitectura de red neuronal convolucional y también hemos presentado varias arquitecturas famosas que han demostrado buenas prestaciones en multitud de tareas. Estas redes están formadas muchas capas convolucionales, algunas con más de 100 capas, lo cual significa que tienen muchos parámetros y entrenarlas desde cero puedes ser costoso. Sin embargo, existe una técnica que nos permite obtener buenos modelos con menores requisitos: el transfer learning. Ya hemos hablado anteriormente de esta técnica, en el contexto de modelos de lenguaje, pero la idea es la misma: utilizaremos el máximo número de capas de una red ya entrenada en otro dataset, y simplemente entrenaremos las nuevas capas que necesitemos para nuestra tarea concreta.

Nuestro objetivo será el de entrenar un clasificador de flores. Podemos descargar las imágenes de la siguiente url.



In [ ]:
import urllib.request, zipfile

urllib.request.urlretrieve('https://mymldatasets.s3.eu-de.cloud-object-storage.appdomain.cloud/flowers.zip', 'flowers.zip')

with zipfile.ZipFile('flowers.zip', 'r') as zip_ref:
    zip_ref.extractall('.')

<a id="section21"></a>
# <font color="#004D7F" size=5>2.1. Preparación del dataset</font>

Una vez extraído el dataset, podemos ver que tenemos 5 clases de flores diferentes, distribuidas en 5 carpetas diferentes. Cada carpeta contiene varios ejemplos de flores de la categoría en cuestión.

In [ ]:
import os 

PATH = 'flowers'

classes = os.listdir(PATH)
classes

In [ ]:
imgs, labels = [], []

for i, lab in enumerate(classes):
  paths = os.listdir(f'{PATH}/{lab}')
  print(f'Categoría: {lab}. Imágenes: {len(paths)}')
  paths = [p for p in paths if p[-3:] == "jpg"]
  imgs += [f'{PATH}/{lab}/{img}' for img in paths]
  labels += [i]*len(paths)

Podemos visualizar algunas imágenes en el dataset.

In [ ]:
import random 
from PIL import Image
import matplotlib.pyplot as plt

fig, axs = plt.subplots(3,5, figsize=(10,6))
for _ax in axs:
  for ax in _ax:
    ix = random.randint(0, len(imgs)-1)
    img = Image.open(imgs[ix]).convert('RGB')
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(classes[labels[ix]])
plt.show()

<a id="section22"></a>
# <font color="#004D7F" size=5>2.2. Procesamiento de datos</font>

Vamos a crear también un subconjunto de test para poder comparar varios modelos.

In [ ]:
from sklearn.model_selection import train_test_split

train_imgs, test_imgs, train_labels, test_labels = train_test_split(imgs, labels, test_size=0.2, stratify=labels)

len(train_imgs), len(test_imgs)

Y por último creamos nuestros objetos `Dataset` y `DataLoader` para poder darle las imágenes a nuestros modelos.

In [ ]:
import torch

# Configurar el dispositivo
if torch.cuda.is_available():
    device = torch.device("cuda")  # Usar GPU con CUDA
elif torch.backends.mps.is_available():
    device = torch.device("mps")   # Usar GPU con Metal (macOS)
else:
    device = torch.device("cpu")   # Usar CPU como alternativa

print(f"Usando el dispositivo: {device}")

In [ ]:
# Clase Dataset corregida
class Dataset(torch.utils.data.Dataset):
    def __init__(self, X, y, trans=None, device="cpu"):
        self.X = X
        self.y = y
        self.trans = trans
        self.device = device

    def __len__(self):
        return len(self.X)

    def __getitem__(self, ix):
        # Cargar la imagen con PIL (convert('RGB') asegura 3 canales)
        img = Image.open(self.X[ix]).convert('RGB')

        # Aplicar transformaciones
        if self.trans:
            img = self.trans(img)

        # Convertir imagen y etiqueta a tensores
        # se devuelven en CPU: fit() mueve cada lote a la GPU
        # (pin_memory=True falla si los tensores ya están en CUDA)
        img_tensor = T.ToTensor()(img)
        label_tensor = torch.tensor(self.y[ix])

        return img_tensor, label_tensor

Nos aseguraremos que todas las imágenes del dataset tengan las mismas dimensiones: 224x224 píxeles.

In [ ]:
# Transformaciones con torchvision
from PIL import Image
import torchvision.transforms as T

trans = T.Resize((224, 224))

# Crear los datasets
dataset = {
    'train': Dataset(train_imgs, train_labels, trans, device),
    'test': Dataset(test_imgs, test_labels, trans, device),
}

print(f"Train dataset size: {len(dataset['train'])}")
print(f"Test dataset size: {len(dataset['test'])}")

In [ ]:
fig, axs = plt.subplots(3, 5, figsize=(10, 6))
for _ax in axs:
    for ax in _ax:
        ix = random.randint(0, len(dataset['train']) - 1)
        img, lab = dataset['train'][ix]

        # Mover la imagen a la CPU antes de visualizarla
        img = img.cpu()

        ax.imshow(img.permute(1, 2, 0))  # Cambiar formato [C, H, W] a [H, W, C]
        ax.axis('off')
        ax.set_title(classes[lab.item()])  # Convertir el tensor de etiqueta a int
plt.tight_layout()
plt.show()

In [ ]:
dataloader = {
    'train': torch.utils.data.DataLoader(dataset['train'], batch_size=64, shuffle=True, pin_memory=True), 
    'test': torch.utils.data.DataLoader(dataset['test'], batch_size=256, shuffle=False)
}

imgs, labels = next(iter(dataloader['train']))
imgs.shape

---
<div style="text-align: right"> <font size=5> <a href="#indice"><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#004D7F"></i></a></font></div>

---

<a id="section3"></a>
# <font color="#004D7F" size=6>3. ResNet</font>

Vamos a escoger la arquitectura resnet, de la que ya hablamos en el post anterior, para hacer nuestro clasificador. De este modelo usarmos todas las capas excepto la última, la cual sustituiremos por una nueva capa lineal para llevar a cabo la clasificación en 5 clases.

<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i>
Todos los modelos [preentrandos](https://pytorch.org/vision/0.20/models.html)


<div class="alert alert-block alert-info">
    
<i class="fa fa-info-circle" aria-hidden="true"></i>
Información sobre [Modelos y pesos preentrenados](https://pytorch.org/vision/0.20/models.html)


<a id="section31"></a>
# <font color="#004D7F" size=5>3.1. Cargar la red</font>

Vamos a ver el proceso para cargar dicha red, ver su estructura y utilizarla para realizar predicciones.

In [ ]:
import torchvision
# API actual: weights=... (el antiguo pretrained=True está obsoleto)
resnet = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.DEFAULT)
resnet

In [ ]:
class Model(torch.nn.Module):
  def __init__(self, n_outputs=5, pretrained=False, freeze=False):
    super().__init__()
    # descargamos resnet
    resnet = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.DEFAULT if pretrained else None)
    # nos quedamos con todas las capas menos la última
    self.resnet = torch.nn.Sequential(*list(resnet.children())[:-1])
    if freeze:
      for param in self.resnet.parameters():
        param.requires_grad = False  # congelar: no se actualizan estos pesos
    # añadimos una nueva capa lineal para llevar a cabo la clasificación
    self.fc = torch.nn.Linear(512, n_outputs)

  def forward(self, x):
    x = self.resnet(x)
    x = x.view(x.shape[0], -1)
    x = self.fc(x)
    return x

  def unfreeze(self):
    for param in self.resnet.parameters():
        param.requires_grad = True

In [ ]:
model = Model()
outputs = model(torch.randn(64, 3, 224, 224))
outputs.shape

Realizamos el ciclo de entrenamiento.

In [ ]:
from tqdm import tqdm
import numpy as np

def fit(model, dataloader, epochs=5, lr=1e-2):
    model.to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    criterion = torch.nn.CrossEntropyLoss()
    for epoch in range(1, epochs+1):
        model.train()
        train_loss, train_acc = [], []
        bar = tqdm(dataloader['train'])
        for batch in bar:
            X, y = batch
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()
            y_hat = model(X)
            loss = criterion(y_hat, y)
            loss.backward()
            optimizer.step()
            train_loss.append(loss.item())
            acc = (y == torch.argmax(y_hat, axis=1)).sum().item() / len(y)
            train_acc.append(acc)
            bar.set_description(f"loss {np.mean(train_loss):.5f} acc {np.mean(train_acc):.5f}")
        bar = tqdm(dataloader['test'])
        val_loss, val_acc = [], []
        model.eval()
        with torch.no_grad():
            for batch in bar:
                X, y = batch
                X, y = X.to(device), y.to(device)
                y_hat = model(X)
                loss = criterion(y_hat, y)
                val_loss.append(loss.item())
                acc = (y == torch.argmax(y_hat, axis=1)).sum().item() / len(y)
                val_acc.append(acc)
                bar.set_description(f"val_loss {np.mean(val_loss):.5f} val_acc {np.mean(val_acc):.5f}")
        print(f"Epoch {epoch}/{epochs} loss {np.mean(train_loss):.5f} val_loss {np.mean(val_loss):.5f} acc {np.mean(train_acc):.5f} val_acc {np.mean(val_acc):.5f}")


<a id="section32"></a>
# <font color="#004D7F" size=5>3.2. Entrenar el modelo</font>

En primer lugar vamos a entrenar nuestro modelo desde cero para ver qué métricas podemos obtener.

In [ ]:
model = Model()
fit(model, dataloader)

Como puedes ver es complicado conseguir buenas métricas ya que nuestro dataset es muy pequeño.

---
<div style="text-align: right"> <font size=5> <a href="#indice"><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#004D7F"></i></a></font></div>

---

<a id="section4"></a>
# <font color="#004D7F" size=6>4. Transfer Learning</font>

Ahora vamos a entrenar el mismo caso pero, en este caso, utilizando los pesos pre-entrenados de resnet.

In [ ]:
model = Model(pretrained=True, freeze=True)
fit(model, dataloader)

Como puedes ver no sólo obtenemos un mejor modelo en menos epochs sino que además cada epoch tarda menos en completarse. Esto es debido a que, al no estar entrenando gran parte de la red, los requisitos computacionales se reducen considerablemente. Mejores modelos y entrenados más rápido.

---
<div style="text-align: right"> <font size=5> <a href="#indice"><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#004D7F"></i></a></font></div>

---

<a id="section5"></a>
# <font color="#004D7F" size=6> 5. Finetuning</font>

Todavía podemos mejorar un poco más si, además de utilizar los pesos descargados de Imagenet en resnet, entrenamos también la red completa.

In [ ]:
model = Model(pretrained=True, freeze=False)
fit(model, dataloader)

Es común entrenar primero el modelo sin entrenar la red pre-entrenada durante varias epochs y después seguir entrenando, pero permitiendo ahora la actualización de pesos también en la red pre-entrenada (usualmente con un learning rate más pequeño).

In [ ]:
model = Model(pretrained=True, freeze=True)
fit(model, dataloader)
model.unfreeze()  # descongelar y seguir con un lr más pequeño
fit(model, dataloader, lr=1e-4)

---
<div style="text-align: right"> <font size=5> <a href="#indice"><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#004D7F"></i></a></font></div>

---

<div style="text-align: right"> <font size=5> <a href="#indice"><i class="fa fa-arrow-circle-up" aria-hidden="true" style="color:#004D7F"></i></a></font></div>

---

<div style="text-align: right"> <font size=6><i class="fa fa-coffee" aria-hidden="true" style="color:#004D7F"></i> </font></div>